In [1]:
from rich import prompt
from sklearn.externals.array_api_compat import torch
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


/Users/ivan/Desktop/py/llm_test/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", device_map="auto", attn_implementation="sdpa")
#tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")


In [3]:
#model.save_pretrained("./gpt2", from_pt=True)
#tokenizer.save_pretrained("./gpt2", )

In [4]:
model = AutoModelForCausalLM.from_pretrained("./gpt2", device_map="auto", attn_implementation="sdpa")
tokenizer = AutoTokenizer.from_pretrained("./gpt2")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 196.77it/s]


In [5]:
model = model.to("cpu")

In [6]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [7]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

In [8]:
prompt = "all the things she said"

In [9]:
def prompt_to_ids(_prompt):
    tokens = tokenizer.tokenize(_prompt)
    return torch.tensor(tokenizer.convert_tokens_to_ids(tokens))

In [10]:
def gen_pos_ids(length):
    return model.transformer.wpe(
        torch.arange(length).unsqueeze(0))

In [11]:
def ids_to_x(ids):
    embeds = model.transformer.wte(ids)
    pos_emb = gen_pos_ids(ids.shape[0])
    return embeds + pos_emb

In [12]:
def next_id_from_x(_model, _x):
    logits = _model.lm_head(_x)
    next_logit = logits[0, -1, :]
    return next_logit.argmax().item()

In [13]:
def token_from_id(_tokenizer, next_token_id):
    return _tokenizer.decode([next_token_id])

In [14]:
def split_heads(t, n_heads=12):
    B, T, C = t.shape
    t = t.view(B, T, n_heads, C // n_heads)
    return t.permute(0, 2, 1, 3)            # [B, heads, T, head_dim]

In [15]:
def exec_attention(_x, _block, ):
    qkv = _block.attn.c_attn(_x)
    q, k, v = qkv.split(768, dim=2)

    q = split_heads(q)   # [1, 12, T, 64]
    k = split_heads(k)
    v = split_heads(v)

    d = q.shape[-1] ** 0.5
    attn_w = (q @ k.transpose(-2, -1)) / d


    mask = torch.tril((torch.ones(_x.shape[1],_x.shape[1]).unsqueeze(0).unsqueeze(0)))
    attn_w = attn_w.masked_fill(mask == 0, float('-inf'))
    attn_w = torch.softmax(attn_w, dim=-1)

    attn_out = attn_w @ v
    attn_out = attn_out.permute(0, 2, 1, 3).contiguous()
    attn_out = attn_out.view(1, _x.shape[1], 768)
    attn_out = _block.attn.c_proj(attn_out)

    return  attn_out

In [16]:
def exec_mlp(_x, _block):
    x_fc = _block.mlp.c_fc(_x)
    x_act = _block.mlp.act(x_fc)
    x_proj = _block.mlp.c_proj(x_act)
    return x_proj

In [17]:
def exec_decoder_block(_block, _x):
    residual_1 = _x
    x_ln = _block.ln_1(_x)
    _x = exec_attention(x_ln, _block)
    _x = residual_1 + _x

    residual_2 = _x
    x_ln = _block.ln_2(_x)
    _x = exec_mlp(x_ln, _block)
    _x = residual_2 + _x

    return _x

In [101]:
out_tokens = []
out_ids = []

In [127]:
def pipeline():
    input_ids = torch.cat((prompt_to_ids(prompt), torch.tensor(out_ids, dtype=torch.long)), dim=0)
    x = ids_to_x(input_ids)
    for block in model.transformer.h:
        x = exec_decoder_block(block, x)

    x = model.transformer.ln_f(x)
    next_tok_id = next_id_from_x(model, x)
    out_ids.append(next_tok_id)
    out_tokens.append(token_from_id(tokenizer, next_tok_id))

In [130]:
for i in range(0, 100):
    pipeline()

In [131]:
output = prompt + ''.join(out_tokens)
print(output)

all the things she said.

"I'm not going to lie to you, I'm not going to lie to you," she said. "I'm not going to lie to you. I'm not going to lie to you. I'm not going to lie to you. I'm not going to lie to you. I'm not going to lie to you. I'm not going to lie to you. I'm not going to lie to you. I'm not going to lie to you. I'm not going to lie to you
